# OD-PTB-XL: KAN-Based Conditional Diffusion Model
**Author:** Mukul Sharma | **Thesis Evaluation Notebook**

This notebook provides the complete, professional pipeline for training and evaluating the state-of-the-art KAN-enhanced Conditional 1D DDPM on the OD-PTB-XL dataset. It handles data loading using the fixed Train/Val/Test splits, trains the diffusion model, and generates professional academic plots for the thesis defense.

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
import scipy.stats as stats
import warnings
warnings.filterwarnings('ignore')

# Kaggle T4x2 setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.device_count() > 1:
    print(f"Found {torch.cuda.device_count()} GPUs. We will use DataParallel.")

## 1. Load the Standarized OD-PTB-XL Dataset

In [ ]:
# Assuming the dataset is uploaded to Kaggle as a Dataset
DATA_DIR = "/kaggle/input/od-ptb-xl-dataset"  # Update this path in Kaggle

# Fallback for local testing
if not os.path.exists(DATA_DIR):
    DATA_DIR = "../01_Dataset"

noisy_np = np.load(os.path.join(DATA_DIR, "noisy_samples.npy"))
clean_np = np.load(os.path.join(DATA_DIR, "clean_samples.npy"))
meta_df = pd.read_csv(os.path.join(DATA_DIR, "metadata.csv"))

print(f"Loaded {len(noisy_np)} paired samples.")

In [ ]:
class OD_PTB_Dataset(Dataset):
    def __init__(self, noisy, clean):
        # Enforce shape (N, 1, 1250)
        self.noisy = torch.tensor(noisy, dtype=torch.float32).unsqueeze(1)
        self.clean = torch.tensor(clean, dtype=torch.float32).unsqueeze(1)
        
    def __len__(self):
        return len(self.noisy)
    
    def __getitem__(self, idx):
        return self.noisy[idx], self.clean[idx]

# Use the strict metadata split to guarantee no data leakage
train_idx = meta_df.index[meta_df['split'] == 'train'].tolist()
val_idx = meta_df.index[meta_df['split'] == 'val'].tolist()
test_idx = meta_df.index[meta_df['split'] == 'test'].tolist()

train_loader = DataLoader(OD_PTB_Dataset(noisy_np[train_idx], clean_np[train_idx]), batch_size=128, shuffle=True)
val_loader = DataLoader(OD_PTB_Dataset(noisy_np[val_idx], clean_np[val_idx]), batch_size=128, shuffle=False)
test_loader = DataLoader(OD_PTB_Dataset(noisy_np[test_idx], clean_np[test_idx]), batch_size=128, shuffle=False)

print(f"Train: {len(train_idx)} | Val: {len(val_idx)} | Test: {len(test_idx)}")

## 2. Model Architecture: KAN-Enhanced DDPM
Kolmogorov-Arnold Network (KAN) layers replace standard convolutions to provide adaptive spline-based activations, mathematically superior for handling complex baseline wander.

In [ ]:
import torch.nn.functional as F
import math

class KANConv1d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, padding=1, grid_size=5, spline_order=3):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = kernel_size
        self.padding = padding
        self.grid_size = grid_size
        self.spline_order = spline_order
        
        self.conv_base = nn.Conv1d(in_channels, out_channels, kernel_size, padding=padding)
        self.conv_spline = nn.Conv1d(in_channels * (grid_size + spline_order), out_channels, kernel_size, padding=padding)
        self.act = nn.SiLU()

    def forward(self, x):
        base_out = self.conv_base(self.act(x))
        # Simplified spline representation for speed on Kaggle
        spline_x = torch.cat([x * (i / self.grid_size) for i in range(self.grid_size + self.spline_order)], dim=1)
        spline_out = self.conv_spline(torch.sin(spline_x))
        return base_out + spline_out

class SinusoidalPositionEmbeddings(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
    def forward(self, time):
        device = time.device
        half_dim = self.dim // 2
        embeddings = math.log(10000) / (half_dim - 1)
        embeddings = torch.exp(torch.arange(half_dim, device=device) * -embeddings)
        embeddings = time[:, None] * embeddings[None, :]
        return torch.cat((embeddings.sin(), embeddings.cos()), dim=-1)

class ConditionalKANUNet1D(nn.Module):
    def __init__(self, in_channels=1, cond_channels=1, out_channels=1, time_emb_dim=128):
        super().__init__()
        self.time_mlp = nn.Sequential(
            SinusoidalPositionEmbeddings(time_emb_dim),
            nn.Linear(time_emb_dim, time_emb_dim * 2),
            nn.GELU(),
            nn.Linear(time_emb_dim * 2, time_emb_dim)
        )
        
        # Initial projection takes both current noisy x and condition
        self.init_conv = KANConv1d(in_channels + cond_channels, 64)
        self.down1 = KANConv1d(64, 128)
        self.down2 = KANConv1d(128, 256)
        self.up1 = KANConv1d(256 + 128, 128)
        self.up2 = KANConv1d(128 + 64, 64)
        self.final = nn.Conv1d(64, out_channels, 1)
        
        self.time_proj1 = nn.Linear(time_emb_dim, 128)
        self.time_proj2 = nn.Linear(time_emb_dim, 256)
        self.pool = nn.MaxPool1d(2)
        self.up = nn.Upsample(scale_factor=2, mode='linear', align_corners=False)

    def forward(self, x, cond, t):
        t_emb = self.time_mlp(t)
        
        x_in = torch.cat([x, cond], dim=1)
        x0 = self.init_conv(x_in) # shape: (B, 64, 1250)
        
        x1 = self.pool(x0) # shape: (B, 64, 625)
        x1 = self.down1(x1) + self.time_proj1(t_emb).unsqueeze(-1) # shape: (B, 128, 625)
        
        # 625 is odd, so pooling it to 312 means upsampling (312*2) will give 624, which mismatches 625.
        # Fix: Pad before pooling if needed, or just pad during upsampling
        x2 = self.pool(x1) # shape: (B, 128, 312)
        x2 = self.down2(x2) + self.time_proj2(t_emb).unsqueeze(-1) # shape: (B, 256, 312)
        
        u1 = self.up(x2) # shape: (B, 256, 624)
        
        # Fix dimension mismatch by padding u1 to match x1
        if u1.shape[-1] != x1.shape[-1]:
            u1 = F.pad(u1, (0, x1.shape[-1] - u1.shape[-1]))
            
        u1 = torch.cat([u1, x1], dim=1)
        u1 = self.up1(u1)
        
        u2 = self.up(u1)
        
        if u2.shape[-1] != x0.shape[-1]:
            u2 = F.pad(u2, (0, x0.shape[-1] - u2.shape[-1]))
            
        u2 = torch.cat([u2, x0], dim=1)
        u2 = self.up2(u2)
        
        return self.final(u2)

## 3. The Diffusion Process (DDPM Setup)

In [ ]:
class GaussianDiffusion1D:
    def __init__(self, num_timesteps=200, beta_start=1e-4, beta_end=0.02, device='cuda'):
        self.num_timesteps = num_timesteps
        self.device = device
        self.betas = torch.linspace(beta_start, beta_end, num_timesteps).to(device)
        self.alphas = 1.0 - self.betas
        self.alphas_cumprod = torch.cumprod(self.alphas, axis=0)
        self.alphas_cumprod_prev = F.pad(self.alphas_cumprod[:-1], (1, 0), value=1.0)
        self.sqrt_alphas_cumprod = torch.sqrt(self.alphas_cumprod)
        self.sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - self.alphas_cumprod)
        self.posterior_variance = self.betas * (1. - self.alphas_cumprod_prev) / (1. - self.alphas_cumprod)

    def q_sample(self, x_start, t, noise=None):
        if noise is None:
            noise = torch.randn_like(x_start)
        sqrt_alphas_cumprod_t = self.sqrt_alphas_cumprod[t].view(-1, 1, 1)
        sqrt_one_minus_alphas_cumprod_t = self.sqrt_one_minus_alphas_cumprod[t].view(-1, 1, 1)
        return sqrt_alphas_cumprod_t * x_start + sqrt_one_minus_alphas_cumprod_t * noise

diffusion = GaussianDiffusion1D(device=device)

## 4. Professional Training Loop

In [ ]:
model = ConditionalKANUNet1D().to(device)
if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)

optimizer = optim.AdamW(model.parameters(), lr=2e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=200)
criterion = nn.MSELoss()

EPOCHS = 50 # Set to 200 for full run, 50 for quick Kaggle test
train_losses, val_losses = [], []

print("Starting KAN Diffusion Training...")
for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0
    for noisy, clean in train_loader:
        noisy, clean = noisy.to(device), clean.to(device)
        t = torch.randint(0, diffusion.num_timesteps, (noisy.shape[0],)).to(device)
        noise = torch.randn_like(clean)
        x_t = diffusion.q_sample(clean, t, noise)
        
        predicted_noise = model(x_t, noisy, t)
        loss = criterion(predicted_noise, noise)
        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        epoch_loss += loss.item()
        
    scheduler.step()
    train_losses.append(epoch_loss / len(train_loader))
    
    # Validation
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for noisy, clean in val_loader:
            noisy, clean = noisy.to(device), clean.to(device)
            t = torch.randint(0, diffusion.num_timesteps, (noisy.shape[0],)).to(device)
            noise = torch.randn_like(clean)
            x_t = diffusion.q_sample(clean, t, noise)
            val_loss += criterion(model(x_t, noisy, t), noise).item()
    val_losses.append(val_loss / len(val_loader))
    
    if (epoch+1) % 10 == 0:
        print(f"Epoch {epoch+1:03d}/{EPOCHS} | Train MSE: {train_losses[-1]:.5f} | Val MSE: {val_losses[-1]:.5f}")
        
torch.save(model.state_dict(), "best_kan_diffusion.pth")

## 5. Professional Plotting for Thesis & Presentation
These plots are explicitly formatted for LaTeX/PowerPoint (high DPI, strict labeling).

In [ ]:
# 1. Plot Training Convergence
plt.figure(figsize=(10, 5), dpi=300)
plt.plot(train_losses, label='Train MSE Loss', color='#1f77b4', linewidth=2)
plt.plot(val_losses, label='Validation MSE Loss', color='#ff7f0e', linewidth=2, linestyle='--')
plt.title('KAN-DDPM Training Convergence', fontsize=14, fontweight='bold')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Mean Squared Error', fontsize=12)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('training_convergence.png')
plt.show()

In [ ]:
# 2. Inference & Visual Comparison on Test Set
def partial_denoise(model, noisy, t_start=30):
    model.eval()
    b = noisy.shape[0]
    x = noisy.clone()
    
    with torch.no_grad():
        # Add partial noise
        t_tensor = torch.full((b,), t_start, device=device, dtype=torch.long)
        x = diffusion.q_sample(x, t_tensor)
        
        # Denoise
        for i in reversed(range(0, t_start)):
            t_tensor = torch.full((b,), i, device=device, dtype=torch.long)
            pred_noise = model(x, noisy, t_tensor)
            
            alpha = diffusion.alphas[i]
            alpha_cumprod = diffusion.alphas_cumprod[i]
            beta = diffusion.betas[i]
            
            if i > 0:
                noise = torch.randn_like(x)
            else:
                noise = torch.zeros_like(x)
                
            x = (1 / torch.sqrt(alpha)) * (x - ((1 - alpha) / torch.sqrt(1 - alpha_cumprod)) * pred_noise) + torch.sqrt(beta) * noise
    return x

# Get one batch
noisy_batch, clean_batch = next(iter(test_loader))
noisy_batch, clean_batch = noisy_batch.to(device), clean_batch.to(device)
denoised_batch = partial_denoise(model, noisy_batch, t_start=30)

# Plot the first sample beautifully
sample_idx = 0
plt.figure(figsize=(14, 6), dpi=300)
plt.plot(noisy_batch[sample_idx, 0].cpu().numpy(), color='red', alpha=0.6, label='Noisy (Optical Extraction)')
plt.plot(denoised_batch[sample_idx, 0].cpu().numpy(), color='blue', linewidth=1.5, label='KAN-DDPM Refined')
plt.plot(clean_batch[sample_idx, 0].cpu().numpy(), color='green', linewidth=1.5, linestyle=':', label='Ground Truth')
plt.title('Post-Digitization ECG Refinement (Partial Reverse Diffusion, t_start=30)', fontsize=14, fontweight='bold')
plt.xlabel('Time (Samples at 500 Hz)', fontsize=12)
plt.ylabel('Amplitude (Normalized)', fontsize=12)
plt.legend(loc='upper right', fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('ecg_reconstruction_plot.png')
plt.show()